# 02b – EDA on `mart_sessions`

A first look at the user table that dbt builds (one row per user): what is in it, what is missing,
and the basic numbers. Run this after `dbt run`. All columns are explained in `docs/data_dictionary.md`.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

DATA_FILE = Path("../data/processed/mart_sessions_all.parquet")

if DATA_FILE.exists():
    sessions = pd.read_parquet(DATA_FILE)
else:
    load_dotenv("../.env", override=True)
    engine = create_engine(
        f"postgresql://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASS')}"
        f"@{os.getenv('POSTGRES_HOST')}:{os.getenv('POSTGRES_PORT')}/{os.getenv('POSTGRES_DB')}"
    )
    sessions = pd.read_sql("SELECT * FROM otto.mart_sessions", engine)
    sessions.to_parquet(DATA_FILE, index=False)

print(sessions.shape)
sessions.head()

(1000000, 20)


,session_id,start_date,start_weekday,start_hour,n_clicks,n_carts,n_orders,n_products,active_days,duration_hours,has_cart,has_order,is_abandoned,hours_cart_to_order,order_weekday,order_hour,is_weekend_order,starts_with_cart_or_order,cart_in_last_days,buyer_type
0,1545849,2022-08-02,Tuesday,13,14,5,6,13,5,580.15,True,True,False,0.21,Tuesday,13.0,False,False,False,quick_buyer
1,1545872,2022-08-02,Tuesday,13,15,0,0,11,7,416.22,False,False,False,NaN,NaN,NaN,None,False,None,browser
2,1545875,2022-08-02,Tuesday,13,154,20,0,119,10,577.31,True,False,True,NaN,NaN,NaN,None,False,False,abandoner
3,1545900,2022-08-02,Tuesday,13,5,0,0,3,1,0.03,False,False,False,NaN,NaN,NaN,None,False,None,browser
4,1545923,2022-08-02,Tuesday,13,74,3,0,35,8,593.20,True,False,True,NaN,NaN,NaN,None,False,False,abandoner


## 1. Columns and missing values

Some columns are empty on purpose: for example `hours_cart_to_order` only has a value for users who ordered after a cart add.

In [2]:
missing = pd.DataFrame({
    "type": sessions.dtypes.astype(str),
    "missing": sessions.isna().sum(),
    "missing_%": (sessions.isna().mean() * 100).round(1),
})
missing

,type,missing,missing_%
session_id,int64,0,0.0
start_date,object,0,0.0
start_weekday,str,0,0.0
start_hour,int64,0,0.0
n_clicks,int64,0,0.0
n_carts,int64,0,0.0
n_orders,int64,0,0.0
n_products,int64,0,0.0
active_days,int64,0,0.0
duration_hours,float64,0,0.0


## 2. Summary statistics

In [3]:
features = ["n_clicks", "n_carts", "n_orders", "n_products", "active_days", "duration_hours", "hours_cart_to_order"]
sessions[features].describe().round(1)

,n_clicks,n_carts,n_orders,n_products,active_days,duration_hours,hours_cart_to_order
count,1000000.0,1000000.0,1000000.0,1000000.0,1000000.0,1000000.0,120978.0
mean,15.2,1.3,0.4,10.4,2.7,164.9,36.5
std,30.2,4.1,1.6,19.1,2.8,202.5,97.2
min,1.0,0.0,0.0,1.0,1.0,0.0,0.0
25%,3.0,0.0,0.0,2.0,1.0,0.2,0.1
50%,5.0,0.0,0.0,4.0,2.0,52.0,0.2
75%,14.0,1.0,0.0,10.0,3.0,310.5,5.0
max,486.0,230.0,98.0,484.0,28.0,672.0,663.1


The mean is much higher than the median for most columns: a few very active users pull the mean up.
That's why the median is used in the analysis.

## 3. Groups

In [4]:
print(sessions["buyer_type"].value_counts())
print()
print(sessions["start_weekday"].value_counts())

buyer_type
browser           702220
abandoner         159946
quick_buyer        81723
slow_buyer         22992
same_day_buyer     16263
late_cart          11668
other               5188
Name: count, dtype: int64

start_weekday
Monday       185807
Tuesday      159950
Wednesday    145137
Sunday       137075
Thursday     131628
Friday       121120
Saturday     119283
Name: count, dtype: int64
